# 05: 成功した動作を集める

SmolVLAを使って、黒いボウルを皿に置くタスクの学習データを集めます。

## Colabの準備

1. Google Driveの「マイドライブ」に `vla_workshop` フォルダを作り、その中に `05_collection` をフォルダごと置きます。
2. このノートブックをColabで開き、ランタイムを **T4 GPU** にします。別の場所に置いた場合は、最初のコードセルの `LESSON_DIR` を変更します。
3. 上から順にセルを実行し、何回試して何回成功したかを動画で確認します。

同じフォルダに置くファイル

- `collection.json`
- `common.py`
- `libero_env.py`
- `prepare.py`
- `requirements.txt`
- `robot_io.py`
- `smolvla.py`

In [ ]:
# @title この回の配布ファイルを準備する
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, files

USE_DRIVE = True  # @param {type:"boolean"}
LESSON_DIR = "/content/drive/MyDrive/vla_workshop/05_collection"  # @param {type:"string"}
REQUIRED_FILES = [
    "collection.json",
    "common.py",
    "libero_env.py",
    "prepare.py",
    "requirements.txt",
    "robot_io.py",
    "smolvla.py",
]

if USE_DRIVE:
    drive.mount("/content/drive")
    source = Path(LESSON_DIR)
else:
    # ipynb以外の今回の配布ファイルを、まとめて選んでアップロードします。
    os.chdir("/content")
    files.upload()
    source = Path("/content")
missing = [name for name in REQUIRED_FILES if not (source / name).is_file()]
if missing:
    raise FileNotFoundError(f"配布ファイルが不足: {missing}。LESSON_DIRを確認してください。")
PROJECT = Path("/content/lesson_05")
PROJECT.mkdir(exist_ok=True)
for name in REQUIRED_FILES:
    shutil.copy2(source / name, PROJECT / name)
os.chdir(PROJECT)
print("この回の作業場所:", PROJECT)

## ライブラリの準備

次のセルで、この回に必要なライブラリを準備します。初回は数分かかることがあります。

In [ ]:
# @title ライブラリを準備する
LESSON_ENV = {
    **os.environ,
    "MPLBACKEND": "Agg",
    "PYTHONPATH": str(PROJECT),
    "PYTHONNOUSERSITE": "1",
}
command = [sys.executable, str(PROJECT / "prepare.py")]

subprocess.run(command, env=LESSON_ENV, check=True)
PYTHON = PROJECT / ".venv/bin/python"


def run_python(program):
    """セルに書いたPythonを、この回の専用Pythonで実行する。"""
    last_result = None
    with subprocess.Popen(
        [str(PYTHON), "-u", "-c", program],
        cwd=PROJECT,
        env=LESSON_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            if line.startswith("Results: "):
                last_result = Path(line.removeprefix("Results: ").strip())
        if process.wait():
            raise RuntimeError("実行に失敗しました。直前のエラーを確認してください。")
    return last_result

## モデルを読み込み、行動を予測する処理

ここでは後のセルで使う処理を用意します。モデルを読み込むのは、実習を実行するセルです。

In [ ]:
MODEL_CODE = r"""
import torch
from lerobot.policies.factory import make_pre_post_processors
from lerobot.policies.smolvla import processor_smolvla  # noqa: F401

from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

from smolvla import checkpoint_inputs, observation_batch


def load_model(settings, image_size=256):
    checkpoint, config, vlm_repo = checkpoint_inputs(settings, image_size)
    policy = SmolVLAPolicy.from_pretrained(checkpoint, config=config, strict=True).eval()
    # 正規化・トークン化・行動の逆正規化はチェックポイント付属のものを使います。
    pre, post = make_pre_post_processors(
        config,
        pretrained_path=checkpoint,
        preprocessor_overrides={
            "device_processor": {"device": config.device},
            "tokenizer_processor": {"tokenizer_name": config.vlm_model_name},
        },
        postprocessor_overrides={"device_processor": {"device": "cpu"}},
    )
    return policy, pre, post, vlm_repo


def predict_action(policy, pre, post, obs, instruction, settings):
    batch = observation_batch(obs, instruction, settings)
    with torch.inference_mode():
        action = post(policy.select_action(pre(batch)))
    return action.squeeze(0).detach().cpu().numpy()

"""

## 何回試すかを決める

まずはタスク0を5回試します。成功が3件に届かなければ `N_TRIALS` を `10` に変え、このセルから再実行してください。

初期設定では25行動ごとに予測し直します。試行回数と初期配置をそろえて `N_ACTION_STEPS` を変えると、採用できる成功試行がどう変わるかを比べられます。

In [ ]:
# @title 収集の設定を選ぶ
N_TRIALS = "5"  # @param ["5", "10"]
N_ACTION_STEPS = "ファイルの設定"  # @param ["ファイルの設定", "1", "5", "10", "25", "50"]

settings = json.loads((PROJECT / "collection.json").read_text())
if N_ACTION_STEPS != "ファイルの設定":
    settings["policy"]["n_action_steps"] = int(N_ACTION_STEPS)
first = settings["initial_states"]["collection"][0]
settings["initial_states"]["collection"] = list(range(first, first + int(N_TRIALS)))
(PROJECT / "active_collection.json").write_text(json.dumps(settings))
print("タスク:", settings["task_ids"])
print("開始時の配置:", settings["initial_states"]["collection"])
print("予測し直す間隔:", settings["policy"]["n_action_steps"])

## リジェクションサンプリングで試行を集める

リジェクションサンプリングは、候補をいくつか作り、決めた条件を満たしたものだけを採用する方法です。ここではロボットが動き始めてから試行が終わるまでを1エピソードとし、エピソードごとに採用を決めます。

1. **候補を作る**。物体の初期配置を変えながら、SmolVLAに同じタスクを実行させます。
2. **成否を判定する**。`env.step(action)` でロボットを動かすたびに、`env.check_success()` で黒いボウルを皿に置けたか確かめます。ボウルをつかんだまま試行が終わると未達です。
3. **成功したエピソードを採用する**。最初から成功までの画像・状態・行動を学習用に保存します。未達の試行は確認用の動画を残します。

各ステップで、行動を取る直前の画像と状態を、そのときSmolVLAが選んだ行動とセットで記録します。06では画像・状態・指示文を入力にし、記録した行動を教師データにします。

In [ ]:
result = run_python(
    MODEL_CODE
    + r"""
from pathlib import Path

import imageio.v2 as imageio
import numpy as np

from common import output_directory, read_json, write_json
from libero_env import open_scene, scene_cases
from robot_io import state_vector, validate_action

settings = read_json("active_collection.json")
policy, pre, post, _ = load_model(settings["policy"])
folder = output_directory("collection")
episodes = []

for number, (task, initial) in enumerate(scene_cases(settings, "collection")):
    print(f"試行 {number} を実行中（開始時の配置 {initial}）", flush=True)
    seed = settings["seed"] + task * 10000 + initial
    torch.manual_seed(seed)
    np.random.seed(seed)
    policy.reset()
    pre.reset()
    post.reset()
    env, obs, instruction = open_scene(settings, task, initial, seed)
    path = folder / f"episode_{number:03d}"
    path.mkdir()
    states, actions, front, wrist = [], [], [], []
    try:
        with imageio.get_writer(path / "video.mp4", fps=20) as video:
            for step in range(settings["max_steps"]):
                action = predict_action(policy, pre, post, obs, instruction, settings["policy"])
                action = validate_action(action)
                states.append(state_vector(obs))
                actions.append(action)
                front.append(obs["agentview_image"][::-1, ::-1].copy())
                wrist.append(obs["robot0_eye_in_hand_image"][::-1, ::-1].copy())
                video.append_data(front[-1])
                obs, _, done, _ = env.step(action)
                success = bool(env.check_success())
                if done or success:
                    break
            video.append_data(obs["agentview_image"][::-1, ::-1])
        # 失敗した試行の動画は、どこで失敗したかの確認に使います。
        # 学習用の画像・state・actionは、成功した試行だけ保存します。
        if success:
            np.savez_compressed(
                path / "frames.npz", state=states, action=actions, front=front, wrist=wrist
            )
        outcome = "成功" if success else "未達"
        print(f"試行 {number}: {outcome}（{len(actions)}ステップ）", flush=True)
        episodes.append({
            "episode": number, "task_id": task, "initial_state": initial,
            "instruction": instruction, "steps": len(actions), "success": success,
        })
    finally:
        env.close()
write_json(folder / "collection.json", {"settings": settings, "episodes": episodes})
print(f"Results: {folder}", flush=True)
"""
)

## 成功した試行を確認する

`SUCCESSFUL_EPISODES` には、`success` が `True` の試行番号だけが入ります。たとえば試行0・1・2の判定が「成功・未達・成功」なら、採用する番号は `[0, 2]` です。自分の実行結果でも、採用番号が動画の「成功」ラベルと一致するか確かめてください。

In [ ]:
from IPython.display import Markdown, Video, display

episodes = json.loads((result / "collection.json").read_text())["episodes"]
SUCCESSFUL_EPISODES = [episode["episode"] for episode in episodes if episode["success"]]
print(f"採用する成功試行: {len(SUCCESSFUL_EPISODES)} / {len(episodes)} 件")
print("採用する試行番号:", SUCCESSFUL_EPISODES)
if len(SUCCESSFUL_EPISODES) < 3:
    print("06に進むには成功試行が3件必要です。試行回数を増やしてもう一度試してください。")

for episode in episodes:
    number = episode["episode"]
    status = "成功" if episode["success"] else "未達"
    display(Markdown(f"### 試行 {number}（{status}）"))
    display(Video(str(result / f"episode_{number:03d}/video.mp4"), embed=True))

## 成功した記録をLeRobot形式に変換する

`SUCCESSFUL_EPISODES` に入った記録をLeRobot形式にまとめます。06へ渡すには3件以上必要です。06では最後の1件を検証用に取り分け、残りをLoRAの学習に使います。

In [ ]:
if not SUCCESSFUL_EPISODES:
    raise ValueError("成功した試行がありません。試行回数を増やしてもう一度試してください")
export_settings = {"folder": str(result), "episodes": SUCCESSFUL_EPISODES}
(PROJECT / "export.json").write_text(json.dumps(export_settings))
dataset_result = run_python(r"""
from pathlib import Path

import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from common import read_json, write_json

settings = read_json("export.json")
folder = Path(settings["folder"])
available = {row["episode"]: row for row in read_json(folder / "collection.json")["episodes"]}
accepted = settings["episodes"]
if any(not available.get(episode, {}).get("success") for episode in accepted):
    raise ValueError("成功した試行だけをデータに変換できます")

image = {"dtype": "image", "shape": (256, 256, 3), "names": ["height", "width", "channels"]}
dataset = LeRobotDataset.create(
    repo_id="workshop/collected_libero",
    root=folder / "dataset",
    fps=20,
    features={
        "observation.state": {"dtype": "float32", "shape": (8,), "names": None},
        "action": {"dtype": "float32", "shape": (7,), "names": None},
        "observation.images.image": image,
        "observation.images.image2": image,
    },
    robot_type="libero_panda",
    use_videos=False,
    video_backend="pyav",
)
try:
    for episode in accepted:
        with np.load(folder / f"episode_{episode:03d}" / "frames.npz") as frames:
            state, action = frames["state"], frames["action"]
            front, wrist = frames["front"], frames["wrist"]
            for index in range(len(action)):
                dataset.add_frame({
                    "observation.state": state[index],
                    "action": action[index],
                    "observation.images.image": front[index],
                    "observation.images.image2": wrist[index],
                    "task": available[episode]["instruction"],
                })
        dataset.save_episode()
finally:
    dataset.finalize()
write_json(folder / "dataset/workshop_provenance.json", {
    "episodes": [available[index] for index in accepted],
})
print(f"Results: {folder / 'dataset'}", flush=True)
""")
print("学習用データ:", dataset_result)

## 確認課題

何回試して、何回成功しましたか。成功した動画では、ロボットはボウルをどこにつかんで、どこに置きましたか。未達の試行があれば、どこで動きが変わったかも見比べてください。

## 結果をDriveに保存する

今回の収集結果と変換済みデータを保存します。変換後の試行番号は、採用順に0から振り直します。成功試行を3件以上変換できたら、表示されたデータのパスを06で使います。

In [ ]:
# @title 結果を保存する
from datetime import UTC, datetime
from uuid import uuid4

RESULTS_DIR = "/content/drive/MyDrive/vla_workshop/results"  # @param {type:"string"}
drive.mount("/content/drive")
destination = Path(RESULTS_DIR)
destination.mkdir(parents=True, exist_ok=True)
name = "05_collection_" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
saved = destination / name
shutil.copytree(result, saved)
print("保存先:", saved)
dataset_path = saved / "dataset"
if (dataset_path / "workshop_provenance.json").is_file():
    info = json.loads((dataset_path / "meta/info.json").read_text())
    if info["total_episodes"] >= 3:
        print("06のDATASET_ROOT:", dataset_path)